<a href="https://colab.research.google.com/github/aishuse/Fine-Tuning-TinyLlama-with-QLoRA/blob/main/Fine_Tuning_TinyLlama_with_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory
%cd /content/LLaMA-Factory

!pip install -r requirements.txt -q
!pip install datasets -q
!pip install -e ".[torch,metrics]" -q
!pip install -q bitsandbytes>=0.39.0

In [ ]:
from datasets import load_dataset
import os

raw = load_dataset("yahma/alpaca-cleaned")["train"]  # ~52k rows[web:59]
subset = raw.shuffle(seed=42).select(range(2000))    # 2k for faster Colab runs

os.makedirs("/content/datasets", exist_ok=True)
subset_path = "/content/datasets/alpaca_subset.json"
subset.to_json(subset_path, orient="records", lines=True)
subset_path


In [ ]:
import json, textwrap, pathlib

cfg_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(cfg_path, "r") as f:
    data_cfg = json.load(f)

data_cfg["alpaca_subset"] = {
    "file_name": subset_path,            # absolute path is fine
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "system": None,
        "history": None
    }
}

with open(cfg_path, "w") as f:
    json.dump(data_cfg, f, indent=2)

print("Registered alpaca_subset in dataset_info.json")
print("Total datasets now:", len(data_cfg))


In [ ]:
yaml_content = """
model_name_or_path: TinyLlama/TinyLlama-1.1B-Chat-v1.0

### method
stage: sft
do_train: true
do_eval: true
finetuning_type: lora

### template & context
template: llama2          # good fit for TinyLlama chat formatting [web:63]
cutoff_len: 256

### output
output_dir: /content/LLaMA-Factory/output/tinyllama-alpaca-qlora
overwrite_output_dir: true

### QLoRA / LoRA config
quantization_bit: 4       # QLoRA 4-bit quantization [web:96]
upcast_layernorm: true

lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### training hyperparameters (CPU/Colab-friendly)
### training hyperparameters (GPU / Colab)
learning_rate: 1e-4
num_train_epochs: 3
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
warmup_steps: 50
max_grad_norm: 1.0
fp16: true                 # use float16 on GPU
# no bf16
# no use_cpu
gradient_checkpointing: true
logging_steps: 10



### dataset registered in dataset_info.json
dataset: alpaca_subset     # points to subset_path you registered [web:41]
val_size: 0.1
max_samples: 2000          # cap to your subset size or less

### checkpointing & evaluation
save_strategy: steps
save_steps: 200
save_total_limit: 2
eval_strategy: steps
eval_steps: 200
plot_loss: true

"""

yaml_path = "/content/LLaMA-Factory/train_lora_qlora_tutor.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

yaml_path


In [ ]:
# 2) Install LLaMA-Factory as a package
%cd /content/LLaMA-Factory




In [ ]:
!python -m llamafactory.cli train /content/LLaMA-Factory/train_lora_qlora_tutor.yaml


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/LLaMA-Factory/output/tinyllama-alpaca-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

def chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    text = f"<s> [INST] {prompt} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(chat("Describe the typical work environment of a doctor."))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/LLaMA-Factory/output/tinyllama-alpaca-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

def chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    text = f"<s> [INST] {prompt} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(chat("what is the difference between LoRA and QLoRA in fine tuning"))


In [ ]:
from google.colab import files
!zip -r /content/LLaMA-Factory.zip /content/LLaMA-Factory
files.download("/content/LLaMA-Factory.zip")


In [ ]:
!zip -r /content/datasets.zip /content/datasets
files.download("/content/datasets.zip")

In [ ]:
print(chat("Summarize the plot of 'Romeo and Juliet' in 3 sentences."))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

base_model.eval()

def base_chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):

    text = f"<s> [INST] {prompt} [/INST]"

    inputs = tokenizer(text, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

prompt = "Explain reinforcement learning in simple terms."

print("BASE MODEL")
print(base_chat(prompt))

print("\nFINE-TUNED MODEL")
print(chat(prompt))

In [ ]:
!pip install evaluate rouge_score -q

In [ ]:
from datasets import load_dataset
from evaluate import load

dataset = load_dataset("yahma/alpaca-cleaned")["train"]
eval_dataset = dataset.select(range(100))   # small evaluation set

generated_texts = []
reference_answers = []

for example in eval_dataset:

    prompt = example["instruction"]
    reference = example["output"]

    response = chat(prompt)

    generated_texts.append(response)
    reference_answers.append(reference)

rouge = load("rouge")

result = rouge.compute(
    predictions=generated_texts,
    references=reference_answers
)

print(result)

In [ ]:
base_texts = [base_chat(ex["instruction"]) for ex in eval_dataset]
base_result = rouge.compute(predictions=base_texts, references=reference_answers)
print(base_result)

In [ ]:
!pip install nbstripout